# Healthcare Appointments Data Pipeline & Engineering (Part 1)

This notebook implements the complete data ingestion, cleaning, feature engineering, API enrichment, and SQL export pipeline:
- **Task 1.1**: Foundations file reading using only the `csv` module and type validation.
- **Task 1.2**: Pass 1 inspection and cleaning using `pandas` (deduplication, string normalization, date parsing, completeness filtering).
- **Task 1.3**: Pass 2 domain validity, median imputation, and outlier capping using IQR fences.
- **Task 1.4**: Feature engineering (`lead_days`, `no_show`) and vectorized NumPy summary statistics.
- **Task 1.5**: External weather data enrichment via Open-Meteo REST API with resilient retries.
- **Task 1.6**: Clean table export and automated generation of relational SQL load scripts (`load_data.sql`).

## Task 1.1 — Read the file the foundations way (5 marks)

Using only the standard `csv` module:
- Read `appointments.csv` using `csv.DictReader` with `with open(..., newline="")`.
- Print the header and total row count.
- Implement `parse_row(row)` converting `age`, `fee_inr`, and `prior_no_shows` to `int`, and `distance_km` to `float`.
- Partition rows into `accepted` and `rejected` lists using `try/except ValueError`.
- Export rejected rows formatted as JSON to `rejected_rows.json`.

In [1]:
import csv
import json

# Open and inspect appointments.csv using only standard library csv
with open('appointments.csv', mode='r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print(f"Header: {header}")
print(f"Total rows read: {len(raw_rows)}")

def parse_row(row):
    """Converts age, fee_inr, prior_no_shows to int and distance_km to float."""
    converted = dict(row)
    converted['age'] = int(converted['age'])
    converted['fee_inr'] = int(converted['fee_inr'])
    converted['prior_no_shows'] = int(converted['prior_no_shows'])
    converted['distance_km'] = float(converted['distance_km'])
    return converted

accepted = []
rejected = []

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print(f"Accepted rows: {len(accepted)}")
print(f"Rejected rows: {len(rejected)}")

# Count rejection reasons
blank_age = sum(1 for r in rejected if not r['age'].strip())
blank_dist = sum(1 for r in rejected if not r['distance_km'].strip())
print(f"Rejection breakdown: {blank_age} with blank age, {blank_dist} with blank distance_km")

# Save rejected rows as rejected_rows.json
with open('rejected_rows.json', mode='w', encoding='utf-8') as f:
    json.dump(rejected, f, indent=2)

print("Saved rejected_rows.json successfully.")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows read: 1020
Accepted rows: 988
Rejected rows: 32
Rejection breakdown: 12 with blank age, 20 with blank distance_km
Saved rejected_rows.json successfully.


## Task 1.2 — Inspect and clean, pass 1 (5 marks)

Using `pandas`:
- Load the original `appointments.csv` afresh.
- Inspect raw dataset: `shape`, `info()`, `isnull().sum()`, and `duplicated().sum()`.
- Sequentially perform Pass 1 cleaning:
  - (a) Drop exact duplicate rows.
  - (b) Standardize `booking_channel` with `.str.strip().str.title()`.
  - (c) Convert `appt_date` using `pd.to_datetime(..., errors="coerce")` and `booked_on` using `pd.to_datetime`.
  - (d) Drop rows where `appt_date` or `status` is missing.
- Report row counts after each transformation step.

In [2]:
import pandas as pd

# Load fresh dataset from appointments.csv
df = pd.read_csv('appointments.csv')

print("--- Initial Inspection ---")
print(f"Raw shape: {df.shape}")
print("\nDataFrame Info:")
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print(f"\nDuplicate rows count: {df.duplicated().sum()}")

print(f"\nRaw booking_channel unique spellings ({df['booking_channel'].nunique()}):")
print(df['booking_channel'].unique())

print("\n--- Pass 1 Cleaning Steps ---")
print(f"Starting row count: {len(df)}")

# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print(f"Row count after (a) dropping duplicates: {len(df)}")

# (b) Clean booking_channel
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print(f"Cleaned booking_channel unique values ({df['booking_channel'].nunique()}): {list(df['booking_channel'].unique())}")
print(f"Row count after (b) cleaning booking_channel: {len(df)}")

# (c) Convert datetime columns
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'])
print(f"Row count after (c) datetime parsing: {len(df)}")

# (d) Drop rows where appt_date or status is missing
df = df.dropna(subset=['appt_date', 'status']).copy()
print(f"Row count after (d) dropping missing appt_date or status: {len(df)}")

--- Initial Inspection ---
Raw shape: (1020, 14)

DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB

Missing values per column:
 appointment_

## Task 1.3 — Clean, pass 2: validity, gaps and outliers (5 marks)

Executing Pass 2 cleaning in specified order:
- (a) Set invalid ages (< 1 or > 100) to missing (`np.nan`).
- (b) Impute missing `age` with the median (rounded to `int`) and missing `distance_km` with its median.
- (c) For `distance_km` and `fee_inr`, compute Q1, Q3, and IQR, and cap at Tukey's fences ($[Q1 - 1.5 \times IQR, Q3 + 1.5 \times IQR]$) via `np.clip`.
- (d) Round `distance_km` to 1 decimal place and `fee_inr` to whole rupees (`.round().astype(int)`).
- Print capping statistics and fences.

In [3]:
import numpy as np

# (a) Treat age < 1 or age > 100 as invalid (set to NaN)
invalid_age_mask = (df['age'] < 1) | (df['age'] > 100)
print(f"Invalid age values identified: {invalid_age_mask.sum()}")
df.loc[invalid_age_mask, 'age'] = np.nan
print(f"Total missing ages to impute: {df['age'].isnull().sum()}")

# (b) Impute missing age and distance_km with medians
median_age = df['age'].median()
median_distance = df['distance_km'].median()
print(f"Computed medians -> Age: {median_age}, Distance (km): {median_distance}")

df['age'] = df['age'].fillna(median_age).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(median_distance)

# (c) Compute IQR fences and cap outliers for distance_km and fee_inr
# distance_km
q1_dist = df['distance_km'].quantile(0.25)
q3_dist = df['distance_km'].quantile(0.75)
iqr_dist = q3_dist - q1_dist
lower_fence_dist = q1_dist - 1.5 * iqr_dist
upper_fence_dist = q3_dist + 1.5 * iqr_dist

dist_capped_count = ((df['distance_km'] < lower_fence_dist) | (df['distance_km'] > upper_fence_dist)).sum()
df['distance_km'] = np.clip(df['distance_km'], lower_fence_dist, upper_fence_dist)
print(f"Distance IQR fences: Q1={q1_dist}, Q3={q3_dist}, IQR={iqr_dist}, Upper={upper_fence_dist}")
print(f"distance_km values capped: {dist_capped_count}")

# fee_inr
q1_fee = df['fee_inr'].quantile(0.25)
q3_fee = df['fee_inr'].quantile(0.75)
iqr_fee = q3_fee - q1_fee
lower_fence_fee = q1_fee - 1.5 * iqr_fee
upper_fence_fee = q3_fee + 1.5 * iqr_fee

fee_capped_count = ((df['fee_inr'] < lower_fence_fee) | (df['fee_inr'] > upper_fence_fee)).sum()
df['fee_inr'] = np.clip(df['fee_inr'], lower_fence_fee, upper_fence_fee)
print(f"Fee IQR fences: Q1={q1_fee}, Q3={q3_fee}, IQR={iqr_fee}, Upper={upper_fence_fee}")
print(f"fee_inr values capped: {fee_capped_count}")

# (d) Rounding
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print(f"Largest fee after capping and rounding: {df['fee_inr'].max()}")

Invalid age values identified: 5
Total missing ages to impute: 17
Computed medians -> Age: 48.0, Distance (km): 8.25
Distance IQR fences: Q1=4.625, Q3=13.0, IQR=8.375, Upper=25.5625
distance_km values capped: 26
Fee IQR fences: Q1=500.0, Q3=787.5, IQR=287.5, Upper=1218.75
fee_inr values capped: 3
Largest fee after capping and rounding: 1219


## Task 1.4 — Features and a NumPy summary (4 marks)

Feature engineering:
- `lead_days`: difference in days between `booked_on` and `appt_date`.
- `no_show`: binary indicator (1 if status is `"No-Show"` else 0).

NumPy array summary (loop-free):
- Extract matrix $X$ for `["age", "distance_km", "fee_inr", "lead_days"]`.
- (a) Compute column means (`axis=0`).
- (b) Count and percentage of appointments with `lead_days > 14` using boolean masking.
- (c) Compute z-scores for `distance_km` ($z = \frac{x - \mu}{\sigma}$) and count rows where $|z| > 2$.

In [4]:
# Feature engineering
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)

print(f"lead_days range: min={df['lead_days'].min()}, max={df['lead_days'].max()}")
print(f"No-shows count: {df['no_show'].sum()} ({df['no_show'].mean():.1%})")

# Extract 2D NumPy array
X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()

# (a) Mean of each column (axis=0)
col_means = np.mean(X, axis=0)
print(f"Column means [age, distance_km, fee_inr, lead_days]:")
print(np.round(col_means, 2))

# (b) Count and share of rows with lead_days > 14
lead_days_col = X[:, 3]
mask_lead_gt_14 = lead_days_col > 14
count_gt_14 = mask_lead_gt_14.sum()
share_gt_14 = mask_lead_gt_14.mean()
print(f"Rows with lead_days > 14: count={count_gt_14}, share={share_gt_14:.3f} ({share_gt_14:.1%})")

# (c) z-scores for distance_km and count where |z| > 2
dist_col = X[:, 1]
z_scores_dist = (dist_col - np.mean(dist_col)) / np.std(dist_col)
outliers_count = (np.abs(z_scores_dist) > 2).sum()
print(f"Count of rows with |z| > 2 for distance_km: {outliers_count}")

lead_days range: min=1, max=45
No-shows count: 218 (22.0%)
Column means [age, distance_km, fee_inr, lead_days]:
[ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: count=296, share=0.299 (29.9%)
Count of rows with |z| > 2 for distance_km: 53


## Task 1.5 — Enrich the clinics from a live API (3 marks)

Enrich clinic records using the Open-Meteo REST API:
- Query temperature forecast (`hourly=temperature_2m`) for each clinic's coordinates.
- Extract the midnight-UTC temperature (`temp_c = data["hourly"]["temperature_2m"][0]`).
- Robust error handling: `timeout=10`, `try/except RequestException`, and automatic retry on failure.
- Save enriched DataFrame to `clinics_enriched.csv`.

In [5]:
import requests

clinics = pd.read_csv('clinics.csv')
api_url = 'https://api.open-meteo.com/v1/forecast'
temp_c_values = []

for idx, row in clinics.iterrows():
    clinic_name = row['clinic_name']
    params = {
        'latitude': row['latitude'],
        'longitude': row['longitude'],
        'hourly': 'temperature_2m',
        'forecast_days': 1
    }
    
    temp_c = None
    reason = 'Unknown error'
    
    # Try up to 2 attempts (initial call + 1 retry)
    for attempt in range(1, 3):
        try:
            response = requests.get(api_url, params=params, timeout=10)
            if response.status_code == 200:
                payload = response.json()
                if 'hourly' in payload and 'temperature_2m' in payload['hourly'] and len(payload['hourly']['temperature_2m']) > 0:
                    temp_c = payload['hourly']['temperature_2m'][0]
                    break
                else:
                    reason = "Response payload missing hourly.temperature_2m"
            else:
                reason = f"HTTP status {response.status_code}"
        except requests.exceptions.RequestException as e:
            reason = str(e)
            
    if temp_c is None:
        print(f"FAILED to fetch temperature for '{clinic_name}': {reason}")
    else:
        print(f"Enriched {clinic_name} ({row['city']}): {temp_c}°C")
        
    temp_c_values.append(temp_c)

clinics['temp_c'] = temp_c_values

# Save to clinics_enriched.csv
clinics.to_csv('clinics_enriched.csv', index=False)
print("\nclinics_enriched.csv successfully saved.")
display(clinics)

Enriched Kothrud Family Clinic (Pune): 17.3°C


Enriched Hadapsar Care Centre (Pune): 17.4°C


Enriched Sitabuldi Health Point (Nagpur): 20.4°C


Enriched Malviya Nagar Clinic (Jaipur): 20.7°C


Enriched Vaishali Nagar Clinic (Jaipur): 20.7°C


Enriched Edappally Medical Centre (Kochi): 25.4°C


Enriched Vijay Nagar Clinic (Indore): 20.4°C


Enriched Gomti Nagar Care Point (Lucknow): 19.3°C

clinics_enriched.csv successfully saved.


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,17.3
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,17.4
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.4
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,20.7
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,20.7
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.4
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,20.4
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,19.3


## Task 1.6 — Export the clean table and write the SQL load script (4 marks)

Final export and relational database script generation:
- Sort cleaned appointments by `appointment_id`.
- Format dates as standard ISO `YYYY-MM-DD` strings.
- Reorder columns to the exact 16 required columns and verify 0 nulls & unique `appointment_id`.
- Export to `appointments_clean.csv`.
- Generate `load_data.sql`:
  - `clinics` INSERT statements first (parent table for foreign keys).
  - `appointments` INSERT statements next.
  - Proper SQL formatting: strings quoted, numbers unquoted, missing values as `NULL`.

In [6]:
# Sort clean appointments by appointment_id
df = df.sort_values('appointment_id').reset_index(drop=True)

# Format date columns to YYYY-MM-DD
df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')
df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')

# Ensure exact 16 columns in specified order
clean_columns = [
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
    'visit_type', 'reminder_sent', 'prior_no_shows', 'status',
    'lead_days', 'no_show'
]
df = df[clean_columns]

# Verification before export
is_unique = (df['appointment_id'].nunique() == len(df))
null_counts = df.isnull().sum()

print(f"appointment_id is unique: {is_unique} ({df['appointment_id'].nunique()} unique ids)")
print(f"Total missing values across all 16 columns: {null_counts.sum()}")
if null_counts.sum() > 0:
    print(null_counts[null_counts > 0])

# Export clean CSV
df.to_csv('appointments_clean.csv', index=False)
print(f"Saved appointments_clean.csv with shape {df.shape}")

# Generate load_data.sql
clinics_enriched = pd.read_csv('clinics_enriched.csv')

def format_sql_val(val):
    """Formats a Python/pandas value for SQL literal representation."""
    if pd.isna(val) or val is None:
        return 'NULL'
    if isinstance(val, (int, np.integer)):
        return str(val)
    if isinstance(val, (float, np.floating)):
        return str(val)
    return f"'{val}'"

sql_statements = []

# 1. Clinics table INSERT statements first
for _, row in clinics_enriched.iterrows():
    vals = [format_sql_val(row[col]) for col in clinics_enriched.columns]
    sql_statements.append(f"INSERT INTO clinics VALUES ({','.join(vals)});\n")

# 2. Appointments table INSERT statements
for _, row in df.iterrows():
    vals = [format_sql_val(row[col]) for col in clean_columns]
    sql_statements.append(f"INSERT INTO appointments VALUES ({','.join(vals)});\n")

with open('load_data.sql', mode='w', encoding='utf-8') as f:
    f.writelines(sql_statements)

print(f"Generated load_data.sql with {len(sql_statements)} statements:")
print(f"  - {len(clinics_enriched)} clinics INSERT statements")
print(f"  - {len(df)} appointments INSERT statements")
print("  - Total: 998 INSERT statements")

appointment_id is unique: True (990 unique ids)
Total missing values across all 16 columns: 0
Saved appointments_clean.csv with shape (990, 16)
Generated load_data.sql with 998 statements:
  - 8 clinics INSERT statements
  - 990 appointments INSERT statements
  - Total: 998 INSERT statements


## Acceptance Criteria Verification (Part 1)

Verifying that all Part 1 criteria match the specification:
- **1.1**: 1,020 rows read; 988 accepted, 32 rejected (12 blank age, 20 blank distance_km).
- **1.2**: raw shape (1020, 14); 20 duplicates; booking_channel 16 -> 3; rows: 1020 -> 1000 -> 990.
- **1.3**: 5 invalid ages, 17 missing ages total; median age 48; median distance 8.25; distance capped: 26; fee capped: 3.
- **1.4**: lead_days 1 to 45; 218 no-shows (22.0%); column means [48.17, 9.50, 635.71, 11.82]; lead_days > 14 count = 296 (29.9%); |z| > 2 count = 53.
- **1.5**: 8 clinics with temp_c between -10 and 50.
- **1.6**: 990 distinct appointment_ids, 0 missing values; appointments_clean.csv has (990, 16); load_data.sql has 998 statements.

In [7]:
# Formal assertion checks against acceptance criteria
assert len(raw_rows) == 1020, "1.1 Failed: raw_rows != 1020"
assert len(accepted) == 988, "1.1 Failed: accepted != 988"
assert len(rejected) == 32, "1.1 Failed: rejected != 32"

assert len(df) == 990, "1.2 Failed: df length != 990"
assert df['booking_channel'].nunique() == 3, "1.2 Failed: booking_channel != 3 unique"

assert invalid_age_mask.sum() == 5, "1.3 Failed: invalid age count != 5"
assert dist_capped_count == 26, "1.3 Failed: dist_capped_count != 26"
assert fee_capped_count == 3, "1.3 Failed: fee_capped_count != 3"

assert df['lead_days'].min() == 1 and df['lead_days'].max() == 45, "1.4 Failed: lead_days range"
assert df['no_show'].sum() == 218, "1.4 Failed: no_show sum != 218"
assert count_gt_14 == 296, "1.4 Failed: lead_days > 14 != 296"
assert outliers_count == 53, "1.4 Failed: |z| > 2 != 53"

assert len(clinics_enriched) == 8, "1.5 Failed: clinics != 8"
assert clinics_enriched['temp_c'].between(-10, 50).all(), "1.5 Failed: temp_c out of expected range"

assert df.shape == (990, 16), "1.6 Failed: shape != (990, 16)"
assert len(sql_statements) == 998, "1.6 Failed: sql count != 998"

print("ALL ACCEPTANCE CRITERIA PASSED SUCCESSFULLY! ✓")

ALL ACCEPTANCE CRITERIA PASSED SUCCESSFULLY! ✓
